<h1>Step 1: Understand the problem statement</h1>

<h3>The Wine dataset</h3>
https://archive.ics.uci.edu/ml/datasets/wine

These data are the results of a chemical analysis of wines grown in the same region
in Italy, derived from three different cultivars (grape varieties). The analysis
determined the quantities of 13 constituents found in each of the three types of wine:

1. Alcohol
2. Malic acid
3. Ash
4. Alcalinity of ash
5. Magnesium
6. Total phenols
7. Flavanoids
8. Nonflavanoid phenols
9. Proanthocyanins
10. Colour intensity
11. Hue
12. OD280/OD315 of diluted wines
13. Proline

All 13 attributes are continuous measurements. The final column,
**Customer_Segment**, records which of the three wine types each row really is
(1, 2 or 3) — 178 wines in total.

**The game for this lab:** we hide Customer_Segment from the clustering algorithm,
let K-Means group the wines purely by their chemistry, and only at the end reveal
the true types to see how well the discovered clusters match reality.


<h1> Step 2 : Import the libraries and dataset </h1>

In [ ]:
# cap the maths library to one thread — avoids the Windows/MKL
# memory-leak warning in K-Means (must run before sklearn is imported)
import os
os.environ["OMP_NUM_THREADS"] = "1"

# pandas          -> loading and handling the data
# matplotlib      -> plotting (including the 3D plot later)
# StandardScaler  -> put all 13 measurements on the same scale
# KMeans          -> the clustering algorithm
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Read the dataset
data = pd.read_csv("data/Wine.csv")

# Look at the first few rows
data.head()



In [ ]:
# X = the 13 chemical measurements ONLY — we drop Customer_Segment so the
# clustering never sees the answer it is supposed to discover
X = data.drop('Customer_Segment', axis=1)

# standardise the features: Proline runs into the thousands while Hue is
# around 1 — without scaling, Proline would dominate every distance
# K-Means measures and the other 12 columns would barely count
X_scaled = StandardScaler().fit_transform(X)

<h1>Step 3: Determine the optimum number of clusters using the Elbow Method</h1>
We use the same technique as our previous lesson: fit K-Means for k = 1 to 10, record the inertia (WCSS —
the within-cluster sum of squares), and look for the "elbow" where the line stops
dropping steeply.

In [ ]:
# Basically, number of clusters = the x-axis value of the point that is the
# corner of the "elbow" (the plot often looks like an elbow)
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=12, random_state=0)
    kmeans.fit(X_scaled)                # cluster the SCALED measurements
    wcss.append(kmeans.inertia_)        # inertia = total spread within clusters

f3, ax = plt.subplots(figsize=(8, 6))
plt.plot(range(1, 11), wcss)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

<h1>Step 4: Apply K-Means clustering </h1>
The elbow says 3: the curve drops steeply to k=3 (2314 → 1659 → 1278), then the
drops shrink to almost nothing. So we build the model with 3 clusters — which is
promising, given we know there really are 3 wine types.

In [ ]:
# build the model with the output from the elbow method, which is 3
clusterNum = 3
k_means = KMeans(init='k-means++', n_clusters=clusterNum, n_init=12, random_state=0)
k_means.fit(X_scaled)

# labels_ holds the cluster number assigned to each of the 178 wines
labels = k_means.labels_
print(labels)

<h1>Step 5: Add the label</h1>
We attach each wine's cluster number to the dataset, then average the 13
measurements within each cluster — the "chemical fingerprint" of each group.

In [ ]:
# We assign the labels to each row in the dataframe.
data['Clus_km'] = labels
print(data.head())

# the average of each measurement within each cluster —
# the "chemical fingerprint" of each group of wines
print(data.groupby('Clus_km')[X.columns].mean())

<h1>Step 5.1: Trying to see the clusters in 3D</h1>
The wines live in 13 dimensions and we can only plot 3 at a time. Here we pick
three of the more informative measurements — it gives a rough idea, but 10
dimensions are missing from the picture. This is exactly the problem PCA solves next.

In [ ]:
# create a 3-dimensional graph of three chosen measurements, coloured by cluster
f4 = plt.figure(1, figsize=(8, 6))
ax = f4.add_subplot(projection='3d')    # current matplotlib way to make 3D axes
ax.view_init(elev=48, azim=134)         # same viewing angle as the original

ax.set_xlabel('alcohol')
ax.set_ylabel('total Phenols')
ax.set_zlabel('Flavanoids')

ax.scatter(X.iloc[:, 0], X.iloc[:, 5], X.iloc[:, 6], c=labels)
plt.show()

<h1>Step 6: Reduce the 13 dimensions to 2 with PCA</h1>
Principal Component Analysis finds the directions along which the wines vary the
most, and re-expresses each wine using just those directions. We keep the top 2 —
turning every wine's 13 numbers into 2, chosen to preserve as much of the
variation as possible.

In [ ]:
# create a PCA that keeps 2 components and fit it on the scaled measurements
from sklearn.decomposition import PCA
pca_wines = PCA(2)
pca_wines.fit(X_scaled)

In [ ]:
# convert every wine from 13 numbers down to 2
wines_p = pca_wines.transform(X_scaled)

In [ ]:
# plot all 178 wines on the new 2-D map (no clusters yet)
plt.figure(figsize=(8, 6))
plt.scatter(wines_p[:, 0], wines_p[:, 1], color='red')
plt.xlabel('First principal component')
plt.ylabel('Second principal component')

In [ ]:
# the same 178 wines as a table — two columns instead of thirteen
wines_p2 = pd.DataFrame(wines_p)
wines_p2

In [ ]:
# As observed, 3 clusters are ideal
km = KMeans(n_clusters=3, n_init=12, random_state=0)
km

In [ ]:
# cluster the wines using only their 2 principal components,
# and keep the coordinates of the 3 cluster centres
wines_predicted = km.fit_predict(wines_p)
centroids = km.cluster_centers_
centroids

In [ ]:
# how many wines landed in each cluster
pd.Series(wines_predicted).value_counts()

In [ ]:
# attach the new cluster number to the dataframe
data['clusters'] = wines_predicted
data.head()

In [ ]:
# the finished picture: every wine on the 2-D map, coloured by cluster,
# with a black + marking each cluster centre
plt.figure(figsize=(8, 6))
plt.scatter(wines_p[:, 0], wines_p[:, 1], c=wines_predicted, cmap='autumn')
plt.scatter(centroids[:, 0], centroids[:, 1], color='black', marker='+')
plt.xlabel('First principal component')
plt.ylabel('Second principal component')

<h1>What did we learn?</h1>

- **Scale before you cluster.** The 13 measurements are on very different scales, so we
  standardised them first — otherwise the big-number columns decide every distance.
- **The Elbow Method chooses k.** WCSS keeps falling as k grows; the elbow is where it
  stops falling steeply. For this dataset that was 3 clusters.
- **K-Means grouped the wines by their chemistry alone** — no labels needed. Each wine
  received a cluster number, and the group averages showed how the clusters differ.
- **13 dimensions can't be seen.** Plotting three hand-picked columns in 3D only gives a
  partial view — there is no way to look at all 13 at once.
- **PCA solved the viewing problem.** It compressed each wine's 13 numbers down to the
  2 that carry the most variation, and on that 2-D map the clusters are plainly visible.